# Day 2: Embeddings, Walked Through

This notebook walks through embeddings step by step. Run each cell in order.

No installs needed -- everything here is plain Python.

## Cell 1: What are embeddings?

An embedding is a word (or sentence, or document) turned into a list of numbers.

Words with similar meaning end up with similar numbers. That's the whole idea --
computers can't do math on the word "dog", but they can absolutely do math on
`[0.8, 0.9, 0.2, 0.1]`.

Below, we hand-craft a few embeddings ourselves, just to see the shape of the idea
before anything fancy happens.

In [ ]:
# A tiny made-up embedding space: 4 numbers per word.
# We picked these numbers by hand so that animals point in a similar
# direction, and vehicles point in a different similar direction.
WORD_EMBEDDINGS = {
    "cat":    [0.9, 0.8, 0.1, 0.2],
    "dog":    [0.8, 0.9, 0.2, 0.1],
    "kitten": [0.85, 0.75, 0.15, 0.25],
    "car":    [0.1, 0.2, 0.9, 0.8],
    "truck":  [0.2, 0.1, 0.8, 0.9],
}

for word, embedding in WORD_EMBEDDINGS.items():
    print(f"{word:8s} -> {embedding}")

## Cell 2: Show embedding examples

Look at `cat` and `dog` above. Both have a big first number and a big second number,
with small third and fourth numbers.

Now look at `car` and `truck`. They're the opposite: small first two numbers, big last two.

That pattern isn't an accident -- it's what lets us measure similarity with math next.

In [ ]:
# Let's look at just the animal words next to each other.
animal_words = ["cat", "dog", "kitten"]
vehicle_words = ["car", "truck"]

print("Animal embeddings:")
for word in animal_words:
    print(f"  {word:8s} -> {WORD_EMBEDDINGS[word]}")

print("\nVehicle embeddings:")
for word in vehicle_words:
    print(f"  {word:8s} -> {WORD_EMBEDDINGS[word]}")

## Cell 3: Calculate similarity

Now let's put a number on "how similar" two embeddings are, using **cosine similarity**.

Don't let the name scare you -- all it does is measure whether two arrows point in the
same direction, on a scale from -1 (opposite) to 1 (identical direction).

In [ ]:
import math

def dot_product(vector_a, vector_b):
    # Multiply matching positions together and add them all up.
    return sum(a * b for a, b in zip(vector_a, vector_b))

def magnitude(vector):
    # The "length" of the vector, like measuring it with a ruler.
    return math.sqrt(sum(x * x for x in vector))

def cosine_similarity(vector_a, vector_b):
    # How aligned are these two arrows, ignoring their length?
    mag_a = magnitude(vector_a)
    mag_b = magnitude(vector_b)
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot_product(vector_a, vector_b) / (mag_a * mag_b)

print("cat  <-> dog:   ", round(cosine_similarity(WORD_EMBEDDINGS["cat"], WORD_EMBEDDINGS["dog"]), 3))
print("cat  <-> kitten:", round(cosine_similarity(WORD_EMBEDDINGS["cat"], WORD_EMBEDDINGS["kitten"]), 3))
print("car  <-> truck: ", round(cosine_similarity(WORD_EMBEDDINGS["car"], WORD_EMBEDDINGS["truck"]), 3))
print("cat  <-> car:   ", round(cosine_similarity(WORD_EMBEDDINGS["cat"], WORD_EMBEDDINGS["car"]), 3))
print("dog  <-> truck: ", round(cosine_similarity(WORD_EMBEDDINGS["dog"], WORD_EMBEDDINGS["truck"]), 3))

Notice the animal-to-animal and vehicle-to-vehicle scores are close to 1.0,
while animal-to-vehicle scores are much lower. That's cosine similarity doing its job.

## Cell 4: Visualize relationships

Let's expand to more words and see which ones cluster together, using a simple
text-based bar chart. Longer bars mean more similar.

In [ ]:
MORE_WORDS = {
    "cat":    [0.9, 0.8, 0.1, 0.1, 0.1],
    "dog":    [0.8, 0.9, 0.1, 0.2, 0.1],
    "car":    [0.1, 0.1, 0.9, 0.8, 0.1],
    "truck":  [0.1, 0.2, 0.8, 0.9, 0.1],
    "pizza":  [0.1, 0.1, 0.1, 0.1, 0.9],
    "burger": [0.1, 0.15, 0.1, 0.1, 0.85],
}

word_list = list(MORE_WORDS.keys())
pairs = []
for i in range(len(word_list)):
    for j in range(i + 1, len(word_list)):
        a, b = word_list[i], word_list[j]
        similarity = cosine_similarity(MORE_WORDS[a], MORE_WORDS[b])
        pairs.append((a, b, similarity))

pairs.sort(key=lambda item: item[2], reverse=True)

print("Most similar pairs first:\n")
for a, b, similarity in pairs:
    bar = "#" * int(similarity * 30)
    print(f"{a:8s} <-> {b:8s} {bar} {similarity:.3f}")

You should see cat/dog, car/truck, and pizza/burger sitting at the top with the
longest bars -- they cluster with their own category and stay far from the others.

## Cell 5: Compare methods

Now the real test: keyword search vs embedding search, on a query that doesn't
share exact words with the right document.

In [ ]:
documents = [
    {"title": "What is a Vector Database",
     "text": "A vector database stores data as numerical vectors called embeddings. "
             "It allows fast similarity search, so you can find documents with "
             "similar meaning, not just matching keywords."},
    {"title": "Python Data Types",
     "text": "Python has several built-in data types including strings, integers, "
             "floats, lists, dictionaries, tuples, and sets."},
]

query = "How do I search by meaning instead of exact words?"
query_words = set(query.lower().replace("?", "").split())

print("Keyword search (exact word overlap):")
for doc in documents:
    doc_words = set((doc["title"] + " " + doc["text"]).lower().split())
    overlap = query_words & doc_words
    print(f"  {doc['title']:28s} shared words: {overlap if overlap else '(none)'}")

print("\nAn embedding-based search (like comparison.py in this folder) would still")
print("find 'What is a Vector Database' here, because 'search' and 'meaning' map to")
print("the same underlying concept as the document, even without exact word matches.")

## Cell 6: Key takeaways

- An embedding is just a list of numbers representing a word, sentence, or document.
- Similar meaning -> similar numbers. That's the whole trick.
- Cosine similarity measures how close two embeddings are, from -1 to 1.
- Keyword search only finds exact word matches. Embedding search finds matches
  based on meaning, even when the wording is completely different.
- This is exactly why real RAG systems use embeddings for retrieval: users rarely
  phrase their questions the same way the source documents are written.

Next up: how real embedding models generate these numbers, instead of us picking
them by hand.